# Automatic chord recognition — Google Colab



This notebook presents the hierarchical chord-recognition system, reproduces its RWC evaluation, and provides an in-notebook audio player with synchronized chord predictions.



The default system is a hierarchical CQT72 CNN followed by a second-order HSMM. The CNN predicts root, quality, seventh, and extension separately; the HSMM uses the current emissions and the two previous chords to select a musically plausible sequence.



## 1. Colab setup



Choose **Runtime → Run all**. The setup downloads the `submission_test` branch into the temporary Colab runtime and installs the application dependencies into the active Python kernel. No Google Drive mount, uploaded model, local checkout, or virtual environment is required.



The download includes the trained models, cached 15-song evaluation features, continuous CNN posteriors, annotations, and example audio. A fresh Colab runtime must run setup again. CPU works; a GPU can accelerate inference.



In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

BRANCH = "submission_test"
SOURCE_VERSION = "continuous-15-v2"
ARCHIVE_URL = f"https://codeload.github.com/hanneswesterberg/dt2470-project/zip/refs/heads/{BRANCH}"
ROOT = Path(tempfile.gettempdir()) / "dt2470_submission"
marker = ROOT / ".source-branch"

if not marker.is_file() or marker.read_text(encoding="utf-8") != SOURCE_VERSION:
    print("Downloading the submission, models, evaluation data, and example audio …")
    with tempfile.TemporaryDirectory() as directory:
        archive = Path(directory) / "project.zip"
        urllib.request.urlretrieve(ARCHIVE_URL, archive)
        with zipfile.ZipFile(archive) as bundle:
            model_member = next(
                member.filename for member in bundle.infolist()
                if member.filename.endswith("/submission/models/best_model.pt")
            )
            prefix = model_member.removesuffix("models/best_model.pt")
            if ROOT.exists():
                shutil.rmtree(ROOT)
            ROOT.mkdir(parents=True)
            for member in bundle.infolist():
                if member.is_dir() or not member.filename.startswith(prefix):
                    continue
                relative = Path(member.filename[len(prefix):])
                target = (ROOT / relative).resolve()
                if not target.is_relative_to(ROOT.resolve()):
                    raise ValueError(f"Unsafe archive path: {member.filename}")
                target.parent.mkdir(parents=True, exist_ok=True)
                with bundle.open(member) as source, target.open("wb") as destination:
                    shutil.copyfileobj(source, destination)
    marker.write_text(SOURCE_VERSION, encoding="utf-8")

requirements = [
    line for line in (ROOT / "requirements.txt").read_text().splitlines()
    if line.strip() and not line.startswith(("#", "ipykernel", "jupyterlab"))
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--quiet", *requirements],
    check=True,
)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, display

env = os.environ.copy()
env["PYTHONPATH"] = str(ROOT)
env["MPLBACKEND"] = "Agg"
print("Project ready:", ROOT)
print("CNN device:", "cuda" if torch.cuda.is_available() else "cpu")


## 2. Architecture and musical motivation

A chord is not only a single class. Its root describes the harmonic centre, quality describes its basic colour, the seventh changes its function and tension, and extensions add further colour. Predicting these parts with separate but connected heads gives the model useful structure and makes errors easier to interpret.

The CQT keeps pitch relationships visible. The CNN turns short time-frequency windows into probabilities. The second-order HSMM smooths those probabilities over time while modelling chord transitions and durations. This avoids isolated one-frame chord changes and captures common progressions such as tonic–predominant–dominant movement. See `ARCHITECTURE.md` for the complete description.

## 3. Optional training demonstration

The command below is the complete training entry point. It requires the cached CQT feature directory and can take a while, so leave it commented out for a quick presentation.

In [ ]:
# subprocess.run([sys.executable, 'training/train_hierarchical_cnn.py',
#                 '--feature-dir', 'data/features_cqt72_v1',
#                 '--reference-checkpoint', 'models/best_model.pt',
#                 '--split-manifest', 'models/split_manifest.json',
#                 '--epochs', '30'], cwd=ROOT, check=True)

## 4. RWC evaluation

The evaluation uses the fixed 15-song test split in `training/evaluate_rwc.py`. **Frame accuracy** gives every analysis frame equal weight. **Duration-weighted accuracy** gives each frame its actual time span. The first result is a **CNN-only acoustic-model evaluation** on saved annotation-aligned windows. The second compares CNN-only and CNN+HSMM across all 15 songs on the same continuous 0.2-second frames. Those continuous posteriors were generated from the complete audio pipeline and saved so the evaluation remains reproducible without bundling every WAV file. Results are shown at root, root + quality, and root + quality + seventh levels.

In [ ]:
FEATURE_DIR = ROOT / 'features'
assert FEATURE_DIR.is_dir(), f'Feature directory is missing: {FEATURE_DIR}'
subprocess.run(
    [sys.executable, 'training/evaluate_rwc.py',
     '--feature-dir', str(FEATURE_DIR),
     '--checkpoint', 'models/best_model.pt'],
    cwd=ROOT, env=env, check=True,
)
report = json.loads((ROOT / 'output/rwc_test_evaluation.json').read_text())
assert 'continuous_pipeline_comparison' in report, (
    'The downloaded evaluator is outdated. Rerun the setup cell, then rerun this cell.'
)
print(f"Evaluated CNN-only on {len(report['per_song'])} songs.")
print(f"Evaluated CNN+HSMM on {len(report['continuous_pipeline_comparison'])} continuous songs.")

### Aggregate metrics

The first table and chart show all six CNN-only metrics over the fixed 15-song split. The second chart isolates the effect of HSMM sequence decoding across all 15 continuous test songs. Values are macro averages, so every evaluated song contributes equally.

In [ ]:
metric_info = [
    ('root_accuracy', 'Root', 'Frame accuracy'),
    ('root_quality_accuracy', 'Root + quality', 'Frame accuracy'),
    ('root_quality_seventh_accuracy', 'Root + quality + seventh', 'Frame accuracy'),
    ('song_root_percent', 'Root', 'Duration-weighted accuracy'),
    ('song_root_quality_percent', 'Root + quality', 'Duration-weighted accuracy'),
    ('song_root_quality_seventh_percent', 'Root + quality + seventh', 'Duration-weighted accuracy'),
]
metrics = report['cnn_only_macro_average']
aggregate = pd.DataFrame([
    {'Evaluation': evaluation, 'Chord components': components, 'Score': metrics[key]}
    for key, components, evaluation in metric_info
])
display(aggregate.style.hide(axis='index').format({'Score': '{:.2%}'}).bar(subset=['Score'], vmin=0, vmax=1, color='#66c7a5'))

components = ['Root', 'Root + quality', 'Root + quality + seventh']
x = np.arange(len(components))
width = .34
fig, ax = plt.subplots(figsize=(10.5, 5.2), constrained_layout=True)
frame_bars = ax.bar(x - width / 2, 100 * aggregate.iloc[:3]['Score'], width, label='Frame accuracy', color='#5fa8d3')
duration_bars = ax.bar(x + width / 2, 100 * aggregate.iloc[3:]['Score'], width, label='Duration-weighted', color='#e9b949')
ax.bar_label(frame_bars, fmt='%.1f%%', padding=4, fontweight='bold')
ax.bar_label(duration_bars, fmt='%.1f%%', padding=4, fontweight='bold')
ax.set(xticks=x, xticklabels=components, ylim=(0, 100), ylabel='Accuracy (%)',
       title='CNN-only accuracy · fixed 15-song RWC test split')
ax.legend(frameon=False, ncols=2, loc='upper right')
ax.grid(axis='y', alpha=.2)
ax.spines[['top', 'right']].set_visible(False)
plt.show()

continuous = report.get('continuous_pipeline_macro_average', {})
if continuous.get('cnn_only') and continuous.get('cnn_hsmm'):
    comparison_rows = []
    for stage, label in [('cnn_only', 'CNN only'), ('cnn_hsmm', 'CNN + HSMM')]:
        for key, component, evaluation in metric_info:
            comparison_rows.append({'System': label, 'Evaluation': evaluation,
                                    'Chord components': component, 'Score': continuous[stage][key]})
    comparison = pd.DataFrame(comparison_rows)
    display(comparison.style.hide(axis='index').format({'Score': '{:.2%}'}))
    fig, ax = plt.subplots(figsize=(10.5, 5.2), constrained_layout=True)
    cnn_values = [100 * continuous['cnn_only'][key] for key, _, _ in metric_info[3:]]
    hsmm_values = [100 * continuous['cnn_hsmm'][key] for key, _, _ in metric_info[3:]]
    cnn_bars = ax.bar(x - width / 2, cnn_values, width, label='CNN only', color='#5fa8d3')
    hsmm_bars = ax.bar(x + width / 2, hsmm_values, width, label='CNN + HSMM', color='#66c7a5')
    ax.bar_label(cnn_bars, fmt='%.1f%%', padding=4, fontweight='bold')
    ax.bar_label(hsmm_bars, fmt='%.1f%%', padding=4, fontweight='bold')
    compared_songs = len(report.get('continuous_pipeline_comparison', []))
    ax.set(xticks=x, xticklabels=components, ylim=(0, 100), ylabel='Duration-weighted accuracy (%)',
           title=f'Sequence decoding comparison · {compared_songs} continuous RWC test songs')
    ax.legend(frameon=False, ncols=2, loc='upper right')
    ax.grid(axis='y', alpha=.2)
    ax.spines[['top', 'right']].set_visible(False)
    plt.show()

### Per-song metrics

This view exposes all six metrics for every test song, making the spread behind the macro average visible.

In [ ]:
short_names = ['Frame · Root', 'Frame · Root + quality', 'Frame · Complete', 'Duration · Root', 'Duration · Root + quality', 'Duration · Complete']
per_song = pd.DataFrame(
    [[f"RWC P{item['song']:03d}"] + [item['cnn_only'][key] for key, _, _ in metric_info]
     for item in report['per_song']],
    columns=['Song'] + short_names,
).set_index('Song')
display(per_song.style.format('{:.2%}').background_gradient(cmap='YlGn', vmin=0, vmax=1))

order = per_song.iloc[:, -1].sort_values(ascending=False).index
ordered = per_song.loc[order]
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), sharey=True, constrained_layout=True)
for ax, columns, title in zip(axes, [range(3), range(3, 6)], ['Frame accuracy', 'Duration-weighted accuracy']):
    values = 100 * ordered.iloc[:, list(columns)].to_numpy()
    image = ax.imshow(values, cmap='YlGn', vmin=0, vmax=100, aspect='auto')
    ax.set_xticks(range(3), ['Root', 'Root + quality', 'Complete'], rotation=18, ha='right')
    ax.set_yticks(range(len(ordered)), ordered.index)
    ax.set_title(title, fontweight='bold', pad=12)
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            value = values[row, column]
            ax.text(column, row, f'{value:.0f}%', ha='center', va='center', fontsize=8.5,
                    color='white' if value >= 70 else '#132a13', fontweight='bold')
fig.colorbar(image, ax=axes, label='Accuracy (%)', shrink=.82, pad=.025)
fig.suptitle('CNN-only results by test song · sorted by complete duration score', fontsize=14, fontweight='bold')
plt.show()

## 5. Listen to and visualize a song

The default is an annotated RWC song. Inference writes the decoded chord sequence, then the next cell shows a readable 24-second detail view of the waveform, model prediction, and ground truth. The separate player keeps the audio and rolling chord cards synchronized; click any card to seek.

In [ ]:
supported_audio = {'.wav', '.mp3', '.flac', '.ogg'}
audio_files = [path for path in sorted((ROOT / 'audio').glob('*'))
               if path.suffix.lower() in supported_audio and path.stat().st_size > 0]
display(pd.DataFrame({
    'Available song': [path.stem for path in audio_files],
    'Format': [path.suffix.lower().lstrip('.').upper() for path in audio_files],
    'Size (MB)': [path.stat().st_size / 1_000_000 for path in audio_files],
}).style.hide(axis='index').format({'Size (MB)': '{:.1f}'}))

In [ ]:
# Choose any listed song. Annotated RWC examples show ground truth; other songs run prediction-only.
song = 'RWC_P017'
assert any(song in {path.name, path.stem} for path in audio_files), f'Audio not found: {song}'
subprocess.run(
    [sys.executable, 'main.py', '--source', 'local', '--input-dir', 'audio',
     '--cnn-checkpoint', 'models/best_model.pt',
     '--language-model', 'models/harmonic_model.pkl',
     '--split-manifest', 'models/split_manifest.json',
     '--song', song, '--non-interactive'],
    cwd=ROOT, env=env, check=True,
)
summary = json.loads((ROOT / 'output/summary.json').read_text())
result = next(item for item in summary if item['song'] == Path(song).stem)
audio_path = Path(result['audio'])
if not audio_path.is_absolute():
    audio_path = ROOT / audio_path
timeline_path = Path(result['prediction_csv'])
if not timeline_path.is_absolute():
    timeline_path = ROOT / timeline_path
timeline = pd.read_csv(timeline_path)
annotation_path = ROOT / 'annotations' / f'{audio_path.stem}.csv'
ground_truth = None
if annotation_path.is_file():
    ground_truth = pd.read_csv(annotation_path, sep=';').rename(
        columns={'t_start': 'start_seconds', 't_end': 'end_seconds'},
    )
    print(f"Recognized {len(timeline)} chord events and loaded {len(ground_truth)} ground-truth events for {audio_path.name}.")
else:
    print(f"Recognized {len(timeline)} chord events for {audio_path.name}; no ground truth is available.")

In [ ]:
# Show a readable excerpt instead of compressing the entire song into one plot.
waveform, sample_rate = librosa.load(audio_path, sr=8_000, mono=True)
max_points = 60_000
stride = max(1, len(waveform) // max_points)
times = np.arange(0, len(waveform), stride) / sample_rate
duration = len(waveform) / sample_rate

roots = ['C#', 'Eb', 'F#', 'Ab', 'Bb', 'C', 'D', 'E', 'F', 'G', 'A', 'B', 'N']
palette = plt.get_cmap('tab20')(np.linspace(0, 1, len(roots)))
def chord_color(label):
    root = next((root for root in roots[:-1] if str(label).startswith(root)), 'N')
    return palette[roots.index(root)]

view_seconds = min(24, duration)
view_start = min(32, max(0, duration - view_seconds))
view_end = view_start + view_seconds
visible_wave = (times >= view_start) & (times <= view_end)

if ground_truth is not None:
    fig, (wave_ax, predicted_ax, truth_ax) = plt.subplots(3, 1, figsize=(15, 6.8), sharex=True,
        gridspec_kw={'height_ratios': [2.4, 1, 1], 'hspace': .10}, constrained_layout=True)
    plot_axes = (wave_ax, predicted_ax, truth_ax)
    title = f'Readable 24-second detail — {audio_path.stem}'
else:
    fig, (wave_ax, predicted_ax) = plt.subplots(2, 1, figsize=(15, 5.2), sharex=True,
        gridspec_kw={'height_ratios': [2.4, 1], 'hspace': .10}, constrained_layout=True)
    truth_ax = None
    plot_axes = (wave_ax, predicted_ax)
    title = f'Readable 24-second prediction detail — {audio_path.stem}'
wave_ax.plot(times[visible_wave], waveform[::stride][visible_wave], color='#263238', linewidth=.65)
wave_ax.set(ylabel='Amplitude', title=title)
wave_ax.grid(alpha=.16)

label_threshold = .75
for event in timeline.itertuples(index=False):
    start, end, chord = event.start_seconds, event.end_seconds, str(event.chord)
    if end <= view_start or start >= view_end:
        continue
    color = chord_color(chord)
    wave_ax.axvspan(start, end, color=color, alpha=.12, linewidth=0)
    predicted_ax.axvspan(start, end, color=color, alpha=.9, ec='white', lw=.5)
    if min(end, view_end) - max(start, view_start) >= label_threshold:
        predicted_ax.text((max(start, view_start) + min(end, view_end)) / 2, .5, chord, ha='center', va='center',
                      fontsize=10, fontweight='bold')

if ground_truth is not None:
    for event in ground_truth.itertuples(index=False):
        start, end, chord = event.start_seconds, event.end_seconds, str(event.chord)
        if end <= view_start or start >= view_end:
            continue
        display_chord = chord.replace(':maj7', 'maj7').replace(':min7', 'm7').replace(':maj', '').replace(':min', 'm').replace(':', '')
        truth_ax.axvspan(start, end, color=chord_color(display_chord), alpha=.9, ec='white', lw=.5)
        if min(end, view_end) - max(start, view_start) >= label_threshold:
            truth_ax.text((max(start, view_start) + min(end, view_end)) / 2, .5, display_chord,
                          ha='center', va='center', fontsize=10, fontweight='bold')

predicted_ax.set(xlim=(view_start, view_end), ylim=(0, 1), yticks=[])
predicted_ax.set_ylabel('CNN + HSMM', rotation=0, ha='right', va='center')
if truth_ax is not None:
    truth_ax.set(xlim=(view_start, view_end), ylim=(0, 1), yticks=[], xlabel='Time (seconds)')
    truth_ax.set_ylabel('Ground truth', rotation=0, ha='right', va='center')
else:
    predicted_ax.set_xlabel('Time (seconds)')
for axis in plot_axes:
    axis.spines[['top', 'right']].set_visible(False)
plt.show()

### Interactive audio and chord player



The focused player appears directly below. It contains the audio controls, current/previous/next chord, and six rolling chord cards. Ground truth appears inside those elements when an annotation file is available. Audio is compressed and embedded so the player works inside Colab without access to runtime file URLs.



In [ ]:
import base64
import html
import io
import re
import soundfile as sf
from urllib.parse import unquote

viewer_path = (ROOT / "output" / f"{Path(song).stem}_viewer.html").resolve()
assert viewer_path.is_file(), f"Viewer was not generated: {viewer_path}"
viewer_html = viewer_path.read_text(encoding="utf-8")
audio_tag = re.search(r'<audio\b[^>]*\bsrc="([^"]+)"', viewer_html)
assert audio_tag is not None, "Generated viewer has no audio source."
media_path = (viewer_path.parent / unquote(html.unescape(audio_tag.group(1)))).resolve()
assert media_path.is_relative_to(ROOT.resolve()), "Audio source is outside the project."

playback, playback_rate = librosa.load(media_path, sr=22_050, mono=True)
audio_buffer = io.BytesIO()
with sf.SoundFile(
    audio_buffer, mode="w", samplerate=playback_rate, channels=1,
    format="OGG", subtype="VORBIS",
) as compressed:
    for start in range(0, len(playback), 65_536):
        compressed.write(playback[start:start + 65_536])
audio_data = base64.b64encode(audio_buffer.getvalue()).decode("ascii")
viewer_html = viewer_html.replace(
    f'src="{audio_tag.group(1)}"', f'src="data:audio/ogg;base64,{audio_data}"', 1,
)

display(HTML(
    '<iframe title="Synchronized audio and chord player" '
    'sandbox="allow-scripts allow-same-origin" '
    'style="width:100%;height:780px;border:0" '
    f'srcdoc="{html.escape(viewer_html, quote=True)}"></iframe>'
))
